In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [12]:
from pathlib import Path
import gc
import pandas as pd


####Function for Parsing Different Format of Dates
def parse_mixed_datetime(series: pd.Series) -> pd.Series:
    s = series.astype("string").str.replace("\u00a0", " ", regex=False).str.strip()
    iso_mask = s.str.match(r"^\d{4}-\d{2}-\d{2}")
    parsed_iso = pd.to_datetime(s.where(iso_mask), errors="coerce")
    parsed_df  = pd.to_datetime(s.where(~iso_mask), dayfirst=True, errors="coerce")
    return parsed_iso.fillna(parsed_df)


#### Data Read
forecastdemand = pd.read_csv("/content/drive/My Drive/NSW_Electricity_Demand/Data/Raw Data/forecastdemand_nsw.csv", dtype=str)
temprature     = pd.read_csv("/content/drive/My Drive/NSW_Electricity_Demand/Data/Raw Data/temperature_nsw.csv", dtype=str)
demand         = pd.read_csv("/content/drive/My Drive/NSW_Electricity_Demand/Data/Raw Data/totaldemand_nsw.csv", dtype=str)

#### Parsing Date Data
forecastdemand["DATETIME"]    = parse_mixed_datetime(forecastdemand["DATETIME"])
forecastdemand["LASTCHANGED"] = parse_mixed_datetime(forecastdemand["LASTCHANGED"])
temprature["DATETIME"]        = parse_mixed_datetime(temprature["DATETIME"])
demand["DATETIME"]            = parse_mixed_datetime(demand["DATETIME"])


####Numeric Conversion
forecastdemand["PREDISPATCHSEQNO"] = pd.to_numeric(forecastdemand["PREDISPATCHSEQNO"], errors="coerce")
forecastdemand["PERIODID"]         = pd.to_numeric(forecastdemand["PERIODID"], errors="coerce")
forecastdemand["FORECASTDEMAND"]   = pd.to_numeric(forecastdemand["FORECASTDEMAND"], errors="coerce")
demand["TOTALDEMAND"]              = pd.to_numeric(demand["TOTALDEMAND"], errors="coerce")
temprature["TEMPERATURE"]          = pd.to_numeric(temprature["TEMPERATURE"], errors="coerce")





In [13]:
#### Data Preprocessing

#### Forecast Demand Preprocessing
#### Exactly 48 rows per Day
forecastdemand["Time Difference"] = (
    (forecastdemand["DATETIME"] - forecastdemand["LASTCHANGED"]).dt.total_seconds() / 60
).round(0)

forecastdemand = (
    forecastdemand[forecastdemand["LASTCHANGED"] <= forecastdemand["DATETIME"]]
    .sort_values(["DATETIME", "LASTCHANGED"])
    .groupby("DATETIME")
    .tail(1)
    .reset_index(drop=True)
)

#### Forecast Demand Data
forecastdemand.head()
forecastdemand.info()
forecastdemand.describe()
print("Missing values:\n", forecastdemand.isna().sum())
print("\nDuplicate rows:", forecastdemand.duplicated().sum())
print("Shape of dataset:", forecastdemand.shape)
print("Columns:", forecastdemand.columns.tolist())
print("Unique Regions:", forecastdemand['REGIONID'].unique())

#### Temprature Data
temprature.head(5)
print("Shape of dataset:", temprature.shape)
print("\nColumn names:", temprature.columns.tolist())
print("\nData types:\n", temprature.dtypes)
print("\nMissing values:\n", temprature.isna().sum())
print("\nDuplicate rows:", temprature.duplicated().sum())
print("\nSummary statistics:\n", temprature.describe())
temprature = temprature.drop_duplicates().reset_index(drop=True)
print("\nDuplicate rows:", temprature.duplicated().sum())

#### Demand Data

print("Data types:\n", demand.dtypes)
print("\nMissing values:\n", demand.isna().sum())
print("\nDuplicate rows:", demand.duplicated().sum())
print("\nSummary statistics:\n", demand.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 196513 entries, 0 to 196512
Data columns (total 7 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   PREDISPATCHSEQNO  196513 non-null  int64         
 1   REGIONID          196513 non-null  object        
 2   PERIODID          196513 non-null  int64         
 3   FORECASTDEMAND    196513 non-null  float64       
 4   LASTCHANGED       196513 non-null  datetime64[ns]
 5   DATETIME          196513 non-null  datetime64[ns]
 6   Time Difference   196513 non-null  float64       
dtypes: datetime64[ns](2), float64(2), int64(2), object(1)
memory usage: 10.5+ MB
Missing values:
 PREDISPATCHSEQNO    0
REGIONID            0
PERIODID            0
FORECASTDEMAND      0
LASTCHANGED         0
DATETIME            0
Time Difference     0
dtype: int64

Duplicate rows: 0
Shape of dataset: (196513, 7)
Columns: ['PREDISPATCHSEQNO', 'REGIONID', 'PERIODID', 'FORECASTDEMAND', 'LASTCHA

In [16]:
#### Data Merge
#### Sorting the Data Prior for easy Merging
forecastdemand = forecastdemand.sort_values("DATETIME")
demand = demand.sort_values("DATETIME")
temprature = temprature.sort_values("DATETIME")

#### Merge Forecast Demand and Total Demand
fact = pd.merge(
    forecastdemand,
    demand[["DATETIME", "TOTALDEMAND"]],
    on="DATETIME",
    how="left"
)

#### Merge fact and temprature
fact = fact.sort_values("DATETIME").reset_index(drop=True)
temprature = temprature.sort_values("DATETIME").reset_index(drop=True)

# Merge temperature by nearest time within 30 minutes
fact = pd.merge_asof(
    fact,
    temprature[["DATETIME", "TEMPERATURE", "LOCATION"]],
    on="DATETIME",
)

#### Save to parquet format
fact.to_parquet("/content/drive/My Drive/NSW_Electricity_Demand/Data/Processed/fact_merged_clean.parquet", index=False)